# Session 2: the precession formula and a design problem
**A. Real-world problem.** A demonstrator holds a bicycle wheel by one end of its axle. The
wheel is spun and the axle swings slowly round. Can we choose the spin rate so that the axle
makes exactly one full circle every 4 seconds?

**Plan.** (1) moment of inertia of a disc, (2) the rule torque = dL/dt, (3) the formula
`Omega_p = m g r / (I w)`, (4) its assumptions, (5) invert it to choose the spin.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

G = 9.81      # m/s^2

## Step 1: moment of inertia of a solid disc
`I = m R^2 / 2` about its axle (kg m^2). Units: kg times metre squared. Mass far from the
axle counts more (R is squared). A wheel with all its mass on the rim has `I = m R^2`,
twice as much for the same mass and radius.

In [ ]:
def disc_inertia(m, radius):
    """I = m R^2 / 2, kg m^2."""
    return m * radius ** 2 / 2


def ring_inertia(m, radius):
    """Wheel with all its mass on the rim: I = m R^2."""
    return m * radius ** 2


def angular_momentum(inertia, w):
    """L = I w, kg m^2/s."""
    return inertia * w

## Step 2: torque = dL/dt, and the precession rate
Newton's second law for rotation: the torque is the rate of change of the angular momentum
vector. For a wheel with a horizontal axle held at one end, the weight `m g` acts at the
centre, a distance `r` from the support, so `tau = m g r`. It is horizontal and at right
angles to `L`, so it does not change the size of `L`, only its direction. In a short time `dt`
the arrow turns by the angle `d(phi) = tau dt / L`. Dividing by dt:

    Omega_p = d(phi)/dt = tau / L = m g r / (I w)

Assumptions: the spin `w` is fast compared with the circling (`I w` much more than
`sqrt(4 I1 m g r)`, checked below), the spin rate is steady, there is no friction, and
the axle makes a steady angle with the vertical. Units: rad/s. One circle takes `2 pi / Omega_p` seconds.

In [ ]:
def precession_rate(m, r, inertia, w, g=G):
    """Omega_p = m g r / (I w), rad/s."""
    return m * g * r / (inertia * w)


def precession_period(m, r, inertia, w, g=G):
    """Seconds for one full circle of the axle."""
    return 2 * np.pi / precession_rate(m, r, inertia, w, g)


def spin_for_period(m, r, inertia, period, g=G):
    """Spin w (rad/s) that gives one precession circle in `period` seconds (design problem)."""
    return m * g * r * period / (2 * np.pi * inertia)


def rpm(w):
    """rad/s to revolutions per minute."""
    return w * 60 / (2 * np.pi)

## Step 3: when is "fast spin" fast enough?
The exact steady precession of a symmetric top whose axis is at angle `theta` from the
upward vertical (Session 3 derives it from the Lagrangian) solves

    I1 cos(theta) Omega^2 - I3 w3 Omega + m g r = 0

where `I1` is the moment of inertia about a perpendicular axis through the support (parallel
axis theorem: `I1 = m R^2/4 + m r^2` for a disc) and `w3` is the spin component along the axle.
The slow root is the precession we want. It exists only if `(I3 w3)^2 >= 4 I1 m g r cos(theta)`.
At `theta = 90` degrees (horizontal axle) the Omega^2 term vanishes and the formula is exact.

In [ ]:
def inertia_about_support(m, radius, r):
    """I1 = m R^2 / 4 + m r^2 (disc, axis through the support perpendicular to the axle)."""
    return m * radius ** 2 / 4 + m * r ** 2


def precession_exact(m, radius, r, w3, theta, g=G):
    """Slow root of I1 cos(theta) Om^2 - I3 w3 Om + m g r = 0. Returns nan if no steady precession."""
    i3 = disc_inertia(m, radius)
    i1 = inertia_about_support(m, radius, r)
    c = np.cos(theta)
    if abs(c) < 1e-12:
        return m * g * r / (i3 * w3)
    disc = (i3 * w3) ** 2 - 4 * i1 * c * m * g * r
    if disc < 0:
        return float("nan")
    return (i3 * w3 - np.sqrt(disc)) / (2 * i1 * c)


def fast_spin_ratio(m, radius, r, w3, g=G):
    """(I3 w3)^2 / (4 I1 m g r). Values well above 1 mean the fast-spin formula is good."""
    i3 = disc_inertia(m, radius)
    return (i3 * w3) ** 2 / (4 * inertia_about_support(m, radius, r) * m * g * r)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    # the design problem: a 2 kg solid-disc wheel, R = 0.30 m, centre 0.25 m from the support
    m, R, r = 2.0, 0.30, 0.25
    I = disc_inertia(m, R)
    T = 4.0
    w = spin_for_period(m, r, I, T)
    print(f"I = {I:.4f} kg m^2; spin for one circle per {T:.0f} s: {w:.2f} rad/s = {rpm(w):.0f} rpm")
    print(f"check: Omega_p = {precession_rate(m, r, I, w):.4f} rad/s (target {2 * np.pi / T:.4f}), "
          f"period {precession_period(m, r, I, w):.3f} s")
    print(f"L = {angular_momentum(I, w):.3f} kg m^2/s, weight torque = {m * G * r:.3f} N m")
    print(f"fast-spin ratio = {fast_spin_ratio(m, R, r, w):.2f}")
    for k in (0.5, 1, 2):
        print(f"spin x{k}: period {precession_period(m, r, I, k * w):.2f} s")
    for th in (90, 60, 30):
        print(f"axle at {th} deg from vertical, w = {w:.1f}: exact {precession_exact(m, R, r, w, np.radians(th)):.4f} rad/s "
              f"(formula {precession_rate(m, r, I, w):.4f})")

    ws = np.linspace(5, 120, 300)
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(ws, [precession_period(m, r, I, x) for x in ws], label="period 2 pi / Omega_p")
    ax.axhline(T, color="tab:red", ls="--", label="target 4 s")
    ax.plot([w], [T], "o", color="tab:red")
    ax.set_xlabel("spin w (rad/s)")
    ax.set_ylabel("time for one circle (s)")
    ax.set_title("Faster spin, slower precession")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "period_vs_spin.png", dpi=120)

    wl = np.linspace(20, 120, 200)
    fig2, ax2 = plt.subplots(figsize=(6.5, 4.2))
    ax2.plot(wl, [precession_rate(m, r, I, x) for x in wl], label="formula m g r / (I w)")
    ax2.plot(wl, [precession_exact(m, R, r, x, np.radians(60)) for x in wl], "--", label="exact, axle 60 deg from vertical")
    ax2.set_xlabel("spin w (rad/s)")
    ax2.set_ylabel("precession rate (rad/s)")
    ax2.set_title("Formula versus exact steady precession")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "rate_vs_spin.png", dpi=120)

## Coding tasks
1. **Easy.** Use `spin_for_period` to find the spin for a 2 s circle with the same wheel, and check it with `precession_period`. Predict it first.
2. **Stretch.** Use `ring_inertia` instead of `disc_inertia` and re-do the design problem for a 4 s circle; compare with the answer to question type D.
3. **Explain.** Explain why doubling the mass of a solid disc does not change its precession, and give one change that does.

## Assessment
- **A. Conceptual.** Why does a faster spin give a slower precession? And why does the mass not matter for a solid disc of fixed radius?
- **B. Numerical.** A solid-disc wheel with m = 1.0 kg, R = 0.20 m, r = 0.30 m spins at 50 rad/s about a horizontal axle. Find I, Omega_p and the time for one circle.
- **C. Graph and simulation interpretation.** In the period-versus-spin graph (m = 2 kg, R = 0.30 m, r = 0.25 m) the line passes through (34.7, 4). What is its slope, and what would the period be at 17 rad/s?
- **D. Design and optimisation challenge.** Design: a real bicycle wheel has its mass on the rim, so I is about m R^2, twice the disc value. With the same m, R and r, what spin gives a 4 s circle? What stays the same? Which is easier to produce by hand, and what is the trade-off?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    m, R, r = 2.0, 0.30, 0.25
    I = L.disc_inertia(m, R)
    checks.append(("disc: I = m R^2 / 2 = 0.09 kg m^2 for 2 kg and 0.30 m", abs(I - 0.09) < 1e-12))
    checks.append(("ring has twice the inertia of a disc", abs(L.ring_inertia(m, R) - 2 * I) < 1e-12))
    # hand calculation: m g r / (I w) = 2 * 9.81 * 0.25 / (0.09 * 20) = 2.725 rad/s
    checks.append(("hand calculation: 20 rad/s gives 2.725 rad/s", abs(L.precession_rate(m, r, I, 20.0) - 2.725) < 1e-3))
    # worked example: one circle in 4 s needs 34.70 rad/s = 331 rpm
    w = L.spin_for_period(m, r, I, 4.0)
    checks.append((f"worked example: one circle per 4 s needs 34.70 rad/s = 331 rpm (got {w:.2f}, {L.rpm(w):.0f})",
                   abs(w - 34.70) < 0.01 and round(L.rpm(w)) == 331))
    checks.append(("with that spin the period is 4 s", abs(L.precession_period(m, r, I, w) - 4.0) < 1e-9))
    checks.append(("double the spin, double the period", abs(L.precession_period(m, r, I, 2 * w) / 4.0 - 2.0) < 1e-9))
    checks.append(("horizontal axle: exact equals formula", abs(L.precession_exact(m, R, r, w, np.pi / 2) - 2 * np.pi / 4.0) < 1e-9))
    ex = L.precession_exact(m, R, r, w, np.radians(60))
    checks.append(("tilted 60 degrees the exact rate is within 5 percent of the formula here", abs(ex / L.precession_rate(m, r, I, w) - 1) < 0.05))
    big = L.precession_exact(m, R, r, 1000.0, np.radians(60))
    checks.append(("for a much faster spin the formula is better than 0.5 percent", abs(big / L.precession_rate(m, r, I, 1000.0) - 1) < 0.005))
    checks.append(("too slow a spin has no steady precession (nan)", np.isnan(L.precession_exact(m, R, r, 10.0, np.radians(30)))))
    checks.append(("fast-spin ratio is 2.92 for the design", abs(L.fast_spin_ratio(m, R, r, w) - 2.92) < 0.01))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
